## 01 Setup and loading the raw files

In [15]:
import os, json, glob, sqlite3
import numpy as np
import pandas as pd

BASE = ".."
RAW, CLEAN = f"{BASE}/data/raw", f"{BASE}/data/clean"
PARTNERS = ["MX", "CN", "TW", "ROW"]      # ROW = rest of world, built in section 6

frames = []
for path in glob.glob(f"{RAW}/*.json"):
    data = json.load(open(path))
    if not data:
        continue
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.loc[:, ~df.columns.duplicated()]
    _, _, partner, year = os.path.basename(path)[:-5].split("_")
    df["partner"], df["YEAR"] = partner, int(year)
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
raw.shape

(462984, 9)

## 02 Inspecting the raw data

In [16]:
raw.info()
print("\nNulls per column:\n", raw.isna().sum())
print("\nDtypes are all text →", raw.dtypes.unique())
raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 462984 entries, 0 to 462983
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   I_COMMODITY        462984 non-null  str  
 1   I_COMMODITY_SDESC  462984 non-null  str  
 2   GEN_VAL_MO         462984 non-null  str  
 3   MONTH              462984 non-null  str  
 4   YEAR               462984 non-null  int64
 5   COMM_LVL           462984 non-null  str  
 6   partner            462984 non-null  str  
 7   CTY_CODE           330397 non-null  str  
 8   CTY_NAME           330397 non-null  str  
dtypes: int64(1), str(8)
memory usage: 31.8 MB

Nulls per column:
 I_COMMODITY               0
I_COMMODITY_SDESC         0
GEN_VAL_MO                0
MONTH                     0
YEAR                      0
COMM_LVL                  0
partner                   0
CTY_CODE             132587
CTY_NAME             132587
dtype: int64

Dtypes are all text → [<StringDtype(storage='python

,I_COMMODITY,I_COMMODITY_SDESC,GEN_VAL_MO,MONTH,YEAR,COMM_LVL,partner,CTY_CODE,CTY_NAME
0,9812,IMPORTS DUTY FREE UNDER BOND FOR PERMANENT EXH...,7030588,01,2025,HS4,WORLD,NaN,NaN
1,9817,IMPORTS DUTY FREE UNDER SPEC CLASSIF PROV NESOI,272250080,01,2025,HS4,WORLD,NaN,NaN
2,9999,SALVAGE;ESTIMATE OF LOW VALUED IMPORT TRANSACT...,2009305688,01,2025,HS4,WORLD,NaN,NaN
3,8514,"INDUSTRIAL OR LAB ELEC FURNACES ETC, PARTS",108974752,01,2025,HS4,WORLD,NaN,NaN
4,8515,"ELECTRIC, LASER OR OTH LIGHT OR PHOTON BEAM ETC",156288452,01,2025,HS4,WORLD,NaN,NaN


## 03 cleaning: renaming, types, dates, duplicates

In [17]:
df = raw[["partner", "YEAR", "MONTH", "I_COMMODITY", "I_COMMODITY_SDESC", "GEN_VAL_MO"]].rename(columns={
    "YEAR": "year", "MONTH": "month", "I_COMMODITY": "hs4",
    "I_COMMODITY_SDESC": "hs4_desc", "GEN_VAL_MO": "value_usd"})

df["value_usd"] = pd.to_numeric(df["value_usd"])
df["hs4"] = df["hs4"].astype(str).str.zfill(4)
df["month"] = df["month"].astype(int)
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))

key = ["partner", "hs4", "date"]
print("Duplicates on key:", df.duplicated(subset=key).sum())
print("Negative values:", (df["value_usd"] < 0).sum())
df = df.drop_duplicates(subset=key)

Duplicates on key: 0
Negative values: 0


## 04 Joining sectors and building the product dimension

In [18]:
sectors = pd.read_csv(f"{BASE}/data/reference/sector_map.csv", dtype={"hs2": str})
sectors["hs2"] = sectors["hs2"].str.zfill(2)

df["hs2"] = df["hs4"].str[:2]
df = df.merge(sectors[["hs2", "sector"]], on="hs2", how="left")
print("Rows without a sector:", df["sector"].isna().sum())

hs4_dim = (df.sort_values("date")
             .groupby("hs4")
             .agg(hs4_desc=("hs4_desc", "last"), hs2=("hs2", "first"), sector=("sector", "first"))
             .reset_index())
print("Distinct HS4 products:", len(hs4_dim))

Rows without a sector: 0
Distinct HS4 products: 1238


## 05 Real zeros vs missing values

In [19]:
dates = sorted(df["date"].unique())
idx = pd.MultiIndex.from_product([df["partner"].unique(), hs4_dim["hs4"], dates], names=key)
monthly = (df.set_index(key)["value_usd"]
             .reindex(idx, fill_value=0)
             .reset_index())
monthly["year"] = monthly["date"].dt.year
print(f"Explicit rows: {len(df):,} → full grid: {len(monthly):,} (zeros added: {len(monthly) - len(df):,})")

Explicit rows: 462,984 → full grid: 534,816 (zeros added: 71,832)


## 06 Annual table and market shares

In [20]:
annual = (monthly.groupby(["hs4", "year", "partner"])["value_usd"].sum()
                 .unstack("partner")
                 .reset_index())
annual.columns.name = None
annual["ROW"] = annual["WORLD"] - annual[["MX", "CN", "TW"]].sum(axis=1)
print("Negative ROW values (should be 0):", (annual["ROW"] < 0).sum())

for p in PARTNERS:
    annual[f"share_{p}"] = annual[p] / annual["WORLD"].replace(0, np.nan)

annual["period"] = pd.cut(annual["year"], bins=[2016, 2019, 2021, 2025],
                          labels=["baseline", "pandemic", "recent"]).astype(str)
print("Shares > 1 (should be 0):", (annual[[f"share_{p}" for p in PARTNERS]] > 1).sum().sum())
annual.head()

Negative ROW values (should be 0): 0
Shares > 1 (should be 0): 0


,hs4,year,CN,MX,TW,WORLD,ROW,share_MX,share_CN,share_TW,share_ROW,period
0,0101,2017,1169976,11032624,0,667411272,655208672,0.016530,0.001753,0.0,0.981717,baseline
1,0101,2018,230000,13290526,0,575274179,561753653,0.023103,0.000400,0.0,0.976497,baseline
2,0101,2019,0,5022979,0,613556150,608533171,0.008187,0.000000,0.0,0.991813,baseline
3,0101,2020,0,4147109,0,548049895,543902786,0.007567,0.000000,0.0,0.992433,pandemic
4,0101,2021,349654,6725411,0,730975885,723900820,0.009201,0.000478,0.0,0.990321,pandemic


## 07 Features per product: baseline (2017–19) vs recent (2023–25)

In [21]:
cols = PARTNERS + ["WORLD"]
base = annual[annual["period"] == "baseline"].groupby("hs4")[cols].sum()
rec  = annual[annual["period"] == "recent"].groupby("hs4")[cols].sum()

feat = pd.DataFrame(index=base.index.union(rec.index))
for p in PARTNERS:
    feat[f"share_{p}_base"]   = base[p] / base["WORLD"]
    feat[f"share_{p}_recent"] = rec[p] / rec["WORLD"]
    feat[f"d_share_{p}_pp"]   = (feat[f"share_{p}_recent"] - feat[f"share_{p}_base"]) * 100
feat["world_base_bn"]   = base["WORLD"] / 1e9
feat["world_recent_bn"] = rec["WORLD"] / 1e9
feat["world_cagr"]      = (rec["WORLD"] / base["WORLD"]) ** (1 / 6) - 1
feat = feat.replace([np.inf, -np.inf], np.nan)

feat["material"] = feat["world_base_bn"] >= 0.1
feat.index.name = "hs4"
feat = feat.reset_index().merge(hs4_dim, on="hs4", how="left")
print("Products:", len(feat), "| material:", feat["material"].sum())

Products: 1238 | material: 995


## 08 Load: CSV + SQLite

In [22]:
tot_base, tot_rec = base["WORLD"].sum(), rec["WORLD"].sum()
b = base.reindex(feat["hs4"]).fillna(0)
r = rec.reindex(feat["hs4"]).fillna(0)

for p in PARTNERS:
    feat[f"contrib_{p}_pp"] = ((r[p] / tot_rec - b[p] / tot_base) * 100).values
    feat[f"gain_{p}_bn_yr"] = ((r[p] - b[p]) / 3 / 1e9).values

agg = {p: round((rec[p].sum() / tot_rec - base[p].sum() / tot_base) * 100, 2) for p in PARTNERS}
print("Aggregate share change (pp):", agg)
print("Sum of contributions (pp):  ", {p: round(feat[f"contrib_{p}_pp"].sum(), 2) for p in PARTNERS})

show = ["hs4", "hs4_desc", "sector", "contrib_MX_pp", "gain_MX_bn_yr", "contrib_CN_pp", "d_share_MX_pp"]
feat.sort_values("contrib_MX_pp", ascending=False).head(15)[show]

Aggregate share change (pp): {'MX': np.float64(1.36), 'CN': np.float64(-7.08), 'TW': np.float64(1.89), 'ROW': np.float64(3.83)}
Sum of contributions (pp):   {'MX': np.float64(1.36), 'CN': np.float64(-7.08), 'TW': np.float64(1.89), 'ROW': np.float64(3.83)}


,hs4,hs4_desc,sector,contrib_MX_pp,gain_MX_bn_yr,contrib_CN_pp,d_share_MX_pp
1027,8471,AUTOMATIC DATA PROCESS MACHINES; MAGN READER ETC,Machinery,0.557838,42.893996,-0.902727,5.186083
1148,9018,"MEDICAL, SURGICAL, DENTAL OR VET INST, NO ELEC...",Instruments,0.093250,8.936822,0.003448,3.812364
184,2208,"ETHYL ALCOHOL, UNDENAT, UND80% ALC; SPIRIT BEV...",Agri-food,0.082915,4.795222,-0.000554,24.691788
258,2710,OIL (NOT CRUDE) FROM PETROL & BITUM MINERAL ETC.,Minerals & fuels,0.079003,4.348772,-0.011494,4.493359
1047,8504,"ELEC TRANS, STATIC CONV & INDUCT, ADP PWR SUPP...",Electrical & electronics,0.074200,5.049708,-0.100140,3.180968
1108,8708,PARTS & ACCESS FOR MOTOR VEHICLES (HEAD 8701-8...,Vehicles & parts,0.063960,21.163517,-0.112071,4.606865
972,8415,"AIR CONDITIONING MACHINES (TEMP & HUM CHANGE),...",Machinery,0.061563,5.501105,-0.027128,4.789106
1079,8537,"BOARDS, PANELS ETC ELEC SWITCH AND N/C APPAR ETC.",Electrical & electronics,0.049810,5.728159,-0.037925,2.707272
1086,8544,"INSULATED WIRE, CABLE ETC; OPT SHEATH FIB CABLES",Electrical & electronics,0.047737,10.325956,-0.057287,-1.415665
1029,8473,PARTS ETC FOR TYPEWRITERS & OTHER OFFICE MACHINES,Machinery,0.042655,2.219706,-0.316221,1.786817


## 09 HS Revisions

In [23]:
feat["hs_revision_flag"] = (b["WORLD"].values == 0) | (r["WORLD"].values == 0)
print("Products present in only one period (HS revisions):", feat["hs_revision_flag"].sum())
feat.loc[feat["hs_revision_flag"], ["hs4", "hs4_desc", "world_base_bn", "world_recent_bn"]]

Products present in only one period (HS revisions): 10


,hs4,hs4_desc,world_base_bn,world_recent_bn
24,0309,"FLOURS, MEALS AND PELLETS OF FISH, CRUSTACEANS...",0.000000,0.005980
198,2404,TOBACCO PROD FOR INHAL W/O COMB; NICOTINE PROD...,0.000000,2.228527
443,3827,MIX CONT HALOGENAT DERIV OF METHANE ETHN PROPN...,0.000000,0.471309
925,8107,"CADMIUM & ARTICLES THEREOF, INCLUD WASTE & SCRAP",0.004929,0.000000
1041,8485,MACHINES FOR ADDITIVE MANUFACTURING,0.000000,2.056855
1066,8524,"FLAT PANEL DISPLAY MODULES, WH/NOT TOUCH-SENSI...",0.000000,6.238207
1091,8549,ELECTRICAL AND ELECTRONIC WASTE AND SCRAP,0.000000,0.133282
1119,8803,"PARTS OF BALLOONS ETC, AIRCRAFT, SPACECRAFT ETC",56.357812,0.000000
1122,8806,UNMANNED AIRCRAFT,0.000000,1.622849
1123,8807,"PARTS OF GOODS OF HEADING 8801, 8802 OR 8806",0.000000,61.448774


## 10 Load CSV files and SQLite db

In [24]:
os.makedirs(CLEAN, exist_ok=True)
monthly.to_csv(f"{CLEAN}/imports_hs4_monthly.csv.gz", index=False)
annual.to_csv(f"{CLEAN}/imports_hs4_annual.csv", index=False)
feat.to_csv(f"{CLEAN}/hs4_features.csv", index=False)

con = sqlite3.connect(f"{BASE}/data/nearshoring.db")
monthly.to_sql("imports_monthly", con, if_exists="replace", index=False)
annual.to_sql("imports_annual", con, if_exists="replace", index=False)
feat.to_sql("hs4_features", con, if_exists="replace", index=False)
sectors.to_sql("sector_map", con, if_exists="replace", index=False)

pd.read_sql("""
    SELECT year,
           ROUND(SUM(MX)/1e9, 1)  AS mx_bn,
           ROUND(SUM(CN)/1e9, 1)  AS cn_bn,
           ROUND(SUM(TW)/1e9, 1)  AS tw_bn,
           ROUND(SUM(ROW)/1e9, 1) AS row_bn
    FROM imports_annual
    GROUP BY year
    ORDER BY year
""", con)

,year,mx_bn,cn_bn,tw_bn,row_bn
0,2017,312.7,505.2,42.4,1479.3
1,2018,343.7,538.5,45.7,1608.2
2,2019,356.1,449.1,54.2,1632.3
3,2020,323.5,432.5,60.4,1515.0
4,2021,382.6,504.2,77.0,1864.7
5,2022,452.0,536.3,91.7,2159.7
6,2023,472.9,427.2,87.8,2088.6
7,2024,503.1,440.3,116.3,2204.2
8,2025,534.3,308.7,201.4,2370.2


## 11 file sizes

In [25]:
for f in sorted(os.listdir(CLEAN)):
    print(f, round(os.path.getsize(f"{CLEAN}/{f}") / 1e6, 1), "MB")

.gitkeep 0.0 MB
hs4_features.csv 0.6 MB
imports_hs4_annual.csv 1.5 MB
imports_hs4_monthly.csv.gz 3.5 MB
